In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import colorcet as cc
import geopandas as gpd
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)

## Validar relación zonas a microzonas

In [ ]:
from mxcensus.aggregate import load_mg_census

from eodgdl import (
    load_eod,
    load_imeplan_agebs,
    load_mtaz,
    load_taz,
    load_zm_muns,
)

# Output directories for derived artifacts written by this notebook
Path("../outputs/taz_validation").mkdir(parents=True, exist_ok=True)

# Import the survey (read from this repo's data/; call load_eod() to fetch from the mirror)
df_viv, df_hab, df_trips, df_legs = load_eod(Path("../data"))

# Municipios que cubre la encuesta:
muns = load_zm_muns()

# Import survey geometries, provided by IMEPLAN
taz = load_taz(Path("../data/AMG_Zonificacion_para_encuesta.parquet"), drop_ap=True)
mtaz = load_mtaz(taz, Path("../data/AMG_MicroZONAS2023.parquet"), drop_ap=True)
agebs_zones = load_imeplan_agebs(
    taz, Path("../data/RELACION_AGEBS-ZONA_con_datos_censales.parquet"), drop_ap=True
)

# After fix all agebs has a valid microzone and all microzones has at least one ageb
assert len(set(mtaz.index) - set(agebs_zones.MZONA)) == 0
assert len(set(agebs_zones.MZONA) - set(mtaz.index)) == 0

# Marco Geoestadistico + census, fetched from the mxcensus mirror via Pooch
state = 14
mg_aur, mg_loc_ageb = load_mg_census(state=state)
mg_loc_ageb = mg_loc_ageb.query("MUN.isin(@muns.keys())")
mg_aur = mg_aur.query("MUN.isin(@muns.keys())")

In [ ]:
# How many agebs are aggregated into microzones?
agebs_zones.groupby("MZONA").size().value_counts().sort_index().plot.bar()
plt.xlabel("Agebs por microzona.")
plt.ylabel("Conteos");

In [ ]:
from eodgdl import zone_system_report

zone_system_report(taz, mtaz)

Si usamos agebs (urbanas y rurales) como microzonas, tenemos un total de

In [ ]:
mg_aur.shape[0]

de las cuales se distribuyen en

In [ ]:
mg_aur.ADMIN_TYPE.value_counts()

Las AGEB rurales son grandes y están mayormente vacías. Pueden descomponerse en sus localidades rurales y espacio vacío para fines de asignación.

Viviendas por tipo de AGEB:

In [ ]:
# Household distribution in the survey
# I have realized the OD survey always reports agebs, not localities.
# This is wrongly coded in the geometries provided by IMEPLAN
pd.concat(
    [
        df_viv.merge(
            mg_aur[["CVEGEO", "ADMIN_TYPE"]],
            how="left",
            left_on="ageb",
            right_on="CVEGEO",
        )
        .ADMIN_TYPE.value_counts(dropna=False)
        .rename("viv"),
        # Trip distribution in the survey
        df_trips.query("~origen.str.startswith('9')")
        .merge(
            mg_aur[["CVEGEO", "ADMIN_TYPE"]],
            how="left",
            left_on="origen",
            right_on="CVEGEO",
        )
        .ADMIN_TYPE.value_counts(dropna=False)
        .rename("origenes"),
        # Trip distribution in the survey
        df_trips.query("~destino.str.startswith('9')")
        .merge(
            mg_aur[["CVEGEO", "ADMIN_TYPE"]],
            how="left",
            left_on="destino",
            right_on="CVEGEO",
        )
        .ADMIN_TYPE.value_counts(dropna=False)
        .rename("destinos"),
    ],
    axis=1,
)

AGEBS únicas con vivienda en la OD:

In [ ]:
pd.concat(
    [
        df_viv.merge(
            mg_aur[["CVEGEO", "ADMIN_TYPE"]],
            how="left",
            left_on="ageb",
            right_on="CVEGEO",
        )
        .groupby("ageb")
        .ADMIN_TYPE.first()
        .value_counts(dropna=False)
        .rename("viv"),
        # Trip distribution in the survey
        df_trips.query("~origen.str.startswith('9')")
        .merge(
            mg_aur[["CVEGEO", "ADMIN_TYPE"]],
            how="left",
            left_on="origen",
            right_on="CVEGEO",
        )
        .groupby("origen")
        .ADMIN_TYPE.first()
        .value_counts(dropna=False)
        .rename("origenes"),
        # Trip distribution in the survey
        df_trips.query("~destino.str.startswith('9')")
        .merge(
            mg_aur[["CVEGEO", "ADMIN_TYPE"]],
            how="left",
            left_on="destino",
            right_on="CVEGEO",
        )
        .groupby("destino")
        .ADMIN_TYPE.first()
        .value_counts(dropna=False)
        .rename("destinos"),
    ],
    axis=1,
)

In [ ]:
mg_aur["N_VIV_OD"] = (
    mg_aur[["CVEGEO"]]
    .reset_index()
    .merge(df_viv[["ageb"]], how="inner", left_on="CVEGEO", right_on="ageb")
    .groupby(["ENTIDAD", "MUN", "LOC", "AGEB"])
    .size()
    .reindex(mg_aur.index)
    .fillna(0)
)
mg_aur["HAS_VIV_OD"] = mg_aur["N_VIV_OD"] > 0

df_trips_no_access = df_trips.query("~origen.str.startswith('9')")
mg_aur["N_ORIG_OD"] = (
    mg_aur[["CVEGEO"]]
    .reset_index()
    .merge(
        df_trips_no_access[["origen"]], how="inner", left_on="CVEGEO", right_on="origen"
    )
    .groupby(["ENTIDAD", "MUN", "LOC", "AGEB"])
    .size()
    .reindex(mg_aur.index)
    .fillna(0)
)
mg_aur["HAS_ORIG_OD"] = mg_aur["N_ORIG_OD"] > 0

df_trips_no_access = df_trips.query("~destino.str.startswith('9')")
mg_aur["N_DEST_OD"] = (
    mg_aur[["CVEGEO"]]
    .reset_index()
    .merge(
        df_trips_no_access[["destino"]],
        how="inner",
        left_on="CVEGEO",
        right_on="destino",
    )
    .groupby(["ENTIDAD", "MUN", "LOC", "AGEB"])
    .size()
    .reindex(mg_aur.index)
    .fillna(0)
)
mg_aur["HAS_DEST_OD"] = mg_aur["N_DEST_OD"] > 0

mg_aur["IN_OD"] = mg_aur["HAS_VIV_OD"] | mg_aur["HAS_ORIG_OD"] | mg_aur["HAS_DEST_OD"]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
mg_aur.plot(column="HAS_VIV_OD", legend=True, ax=axes[0, 0])
mg_aur.plot(column="HAS_ORIG_OD", legend=True, ax=axes[0, 1])
mg_aur.plot(column="HAS_DEST_OD", legend=True, ax=axes[0, 2])
mg_aur.plot(column="N_VIV_OD", legend=True, ax=axes[1, 0])
mg_aur.plot(column="N_ORIG_OD", legend=True, ax=axes[1, 1])
mg_aur.plot(column="N_DEST_OD", legend=True, ax=axes[1, 2])

In [ ]:
ax = mg_aur.plot(column="IN_OD", legend=True, edgecolor="grey", figsize=(10, 10))
taz.to_crs(mg_aur.crs).plot(ax=ax, facecolor="none")

In [ ]:
ageb_zona_map = (
    pd.concat(
        [
            df_trips[["origen", "zona_origen"]]
            .rename(columns={"origen": "ageb", "zona_origen": "zona"})
            .drop_duplicates()
            .set_index("ageb"),
            df_trips[["destino", "zona_destino"]]
            .rename(columns={"destino": "ageb", "zona_destino": "zona"})
            .drop_duplicates()
            .set_index("ageb"),
        ]
    )
    .reset_index()
    .drop_duplicates()
    .set_index("ageb")
    .to_dict()["zona"]
)

In [ ]:
mg_aur["ZONA"] = mg_aur["CVEGEO"].map(ageb_zona_map)
mg_aur.to_parquet("../outputs/mg_aur.parquet")

In [ ]:
mg_loc = mg_loc_ageb.query("ADMIN_TYPE != 'AGEB_URBAN'")
mg_loc

In [ ]:
mg_loc.PARENT_RURAL_AGEB.unique()

In [ ]:
cvegeo = "140510016"
ar = mg_aur.query("CVEGEO==@cvegeo")
zone = ar.ZONA.item()

ax = ar.plot()
mg_loc.query("PARENT_RURAL_AGEB == @cvegeo").plot(
    ax=ax, color="tab:red", edgecolor="grey"
)
if isinstance(zone, str):
    taz.to_crs(mg_aur.crs).loc[zone:zone].plot(
        ax=ax, facecolor="none", edgecolor="black", lw=2
    )
    agebs_zones.to_crs(mg_aur.crs).query("EOD2023==@zone").plot(
        ax=ax, color="tab:orange", alpha=0.5
    )

In [ ]:
cvegeo = "140510035"
ar = mg_aur.query("CVEGEO==@cvegeo")
zone = ar.ZONA.item()

ax = ar.plot()
mg_loc.query("PARENT_RURAL_AGEB == @cvegeo").plot(
    ax=ax, color="tab:red", edgecolor="grey"
)
if isinstance(zone, str):
    taz.to_crs(mg_aur.crs).loc[zone:zone].plot(
        ax=ax, facecolor="none", edgecolor="black", lw=2
    )

In [ ]:
agebs_zones.query("CVEGEO.str.contains(@cvegeo)")

In [ ]:
zone

In [ ]:
agebs_zones.query("EOD2023==@zone").plot()

In [ ]:
agebs_not_in_mg = agebs_zones[
    agebs_zones.CVEGEO.isin(set(agebs_zones.CVEGEO) - set(mg_loc_ageb.CVEGEO))
][["CVEGEO", "POBTOT"]]
print(
    f"The following agebs are not in Marco Geoestadístico (Censo 2020): {agebs_not_in_mg}"
)

In [ ]:
viv_agebs = df_viv.ageb.unique()
viv_agebs_not_in_implan = viv_agebs[~viv_agebs.isin(agebs_zones.CVEGEO.values)]
len(viv_agebs_not_in_implan)

In [ ]:
pd.Series(list(map(len, viv_agebs))).value_counts()

In [ ]:
viv_agebs_not_in_mg = viv_agebs[~viv_agebs.isin(mg_loc_ageb.CVEGEO.values)]
len(viv_agebs_not_in_mg)

In [ ]:
pd.Series(list(map(len, viv_agebs_not_in_mg))).value_counts()

In [ ]:
# Parece que son agebs urbanas
viv_agebs_not_in_mg

In [ ]:
# Does all trips ends contain an AGEB?
origins = df_trips.origen.unique()
destinies = df_trips.destino.unique()
not_in_agebs_implan = set(origins[~origins.isin(agebs_zones.CVEGEO.values)]).union(
    set(destinies[~destinies.isin(agebs_zones.CVEGEO.values)])
)
print(len(not_in_agebs_implan))

In [ ]:
pd.Series(list(map(len, destinies))).value_counts()

In [ ]:
pd.Series(list(map(len, origins))).value_counts()

In [ ]:
pd.Series(list(map(len, not_in_agebs_implan))).value_counts()

In [ ]:
not_in_mg = set(origins[~origins.isin(mg_loc_ageb.CVEGEO.values)]).union(
    set(destinies[~destinies.isin(mg_loc_ageb.CVEGEO.values)])
)
print(len(not_in_mg))
pd.Series(list(map(len, not_in_mg))).value_counts()

In [ ]:
agebs_zones.loc[agebs_not_in_mg.index][
    [
        "CVEGEO",
        "CVE_ENT",
        "CVE_MUN",
        "CVE_LOC",
        "CVE_AGEB",
        "POBTOT",
        "VIV_ENCUES",
        "PER_ENCUES",
        "EMP_INDUST",
        "EMP_SERVIC",
        "EMP_OTROS",
        "NOM_LOC",
    ]
]

In [ ]:
mg_loc_ageb.loc[14, 97, 852]

In [ ]:
agebs_zones.query("CVEGEO == '141205567'").sjoin(
    mg_loc_ageb.loc[14, 120, :, "0000"].query("CVEGEO.str.contains('5567')")
)[["CVEGEO_right", "POBTOT_right"]]

In [ ]:
agebs_zones.query(
    "CVEGEO.isin(['140970852415A', '1409708524164', '1409708524179','1409708524183', '140970852'])"
)

In [ ]:
ax = agebs_zones.query("CVEGEO.isna()").plot()
# mg_loc_ageb.loc[14, 120, 1].plot(ax=ax, color="green", alpha=0.5)
# mg_loc_ageb.loc[14, 120, :, "0000"].query("CVEGEO.str.contains('5567')").plot(
#     ax=ax, color="red"
# )

In [ ]:
{
    "141200230": ["1412002300034800"],  # Point locality, no jobs
    "1412001790034": ["1412001790034800"],  # Point locality, no jobs
    "1409800011010": [],  # no pob, no jobs
    "1407090134146": ["140700134"],  # pob, population is repeated in target ageb
    "99999000A": ["Airport"],  # no pob, has jobs
    "140970244": ["1409702440104800"],  # point locality, pob, no jobs
    "141200614": ["1412006140123800"],  # point locality, no pob, no jobs
    "141200271": ["1412002710034800"],  # point locality, pob, no jobs
    "141200546": ["1412005460123800"],  # point locality, no pob, no jobs
    "140970852": [
        "140970852415A",
        "1409708524164",
        "1409708524179",
        "1409708524183",
    ],  # pob, no jobs, population from these agebs is repeated
    "140970682": ["1409706820104800"],  # point locality, popualtion, jobs
    "140970809": ["1409708090104800"],  # point locality, population, jobs
    "141205567": [],  # no pob, jobs
    "141011959": [],  # no pob, no jobs
    "1407001161467": [],  # no pob, jobs, penal de puyente grande
    "14070140A": [],  # no pob, jobs, survey
    "141200142": [],  # no pob, jobs, survey
    None: [],  # no pob, no jobs
}

In [ ]:
# Microzones are assigned using area overlaps for polygons
# and within predicate for points
mg_loc_ageb_points = mg_loc_ageb[mg_loc_ageb.geom_type == "MultiPoint"]
mg_loc_ageb_polys = mg_loc_ageb[mg_loc_ageb.geom_type == "MultiPolygon"]
assert len(mg_loc_ageb_polys) + len(mg_loc_ageb_points) == len(mg_loc_ageb)

mzona = (
    mg_loc_ageb_polys.reset_index()[["ENTIDAD", "MUN", "LOC", "AGEB", "geometry"]]
    .assign(AREA=lambda df: df.area)
    .overlay(mtaz.reset_index()[["ID", "geometry"]])
    .assign(FAREA=lambda df: df.area / df.AREA)
    .query("FAREA > 0.8")
    .set_index(["ENTIDAD", "MUN", "LOC", "AGEB"])[["ID"]]
)
mzona.index.is_unique

In [ ]:
agebs_zones.loc[agebs_not_in_mg.index].plot(figsize=(10, 10), column="POBTOT")

In [ ]:
ax = mg_loc_ageb_polys.assign(MZONA=mzona, MZONA_na=lambda df: df.MZONA.isna()).plot(
    column="MZONA_na", figsize=(10, 10)
)
agebs_zones.loc[agebs_not_in_mg.index].plot(
    color="red", ax=ax, alpha=0.5, edgecolor="k"
)
plt.savefig("../outputs/zones.pdf")

In [ ]:
agebs_zones.loc[agebs_not_in_mg.index].plot(color="red")

In [ ]:
# We cannot use IS_URB to filter rural localities
# Some of them are classified differently in the MG
# So we better use AMBITO from MG
mzona = (
    mg_loc_ageb_polys.reset_index()[["ENTIDAD", "MUN", "LOC", "AGEB, geometry"]]
    .assign(AREA=lambda df: df.area)
    .overlay(mtaz.reset_index()[["ID", "geometry"]])
    .assign(FAREA=lambda df: df.area / df.AREA)
    .query("FAREA > 0.8")
    .set_index(["MUN", "LOC"])[["ID"]]
)
assert mzona.index.is_unique
mg_loc_census_rural["MZONA"] = mzona

mg_loc_p_census = mg_loc_p.merge(loc_all, how="left", left_index=True, right_index=True)
mg_loc_p_census.loc[
    mg_loc_p.index[~mg_loc_p.index.isin(loc_all.index)],
    ["POBTOT", "VIVTOT", "TVIVHAB"],
] = 0.0
mzona = mg_loc_p_census[["geometry"]].sjoin(mtaz[["geometry"]])[["ID"]]
assert mzona.index.is_unique
mg_loc_p_census["MZONA"] = mzona

## All agebs are present in mg
assert sum(~mg_ageb.index.isin(agebs.index)) == 0
mg_ageb_census = mg_ageb.merge(
    agebs, how="left", left_index=True, right_index=True
).assign(centroid=lambda df: df.centroid)
mzona = (
    mg_ageb_census.reset_index()[["MUN", "LOC", "AGEB", "geometry"]]
    .assign(AREA=lambda df: df.area)
    .overlay(mtaz.reset_index()[["ID", "geometry"]])
    .assign(FAREA=lambda df: df.area / df.AREA)
    .query("FAREA > 0.8")
    .set_index(["MUN", "LOC", "AGEB"])[["ID"]]
)
assert mzona.index.is_unique
mg_ageb_census["MZONA"] = mzona


In [ ]:
mg_loc_p_census.query("~MZONA.isna()").POBTOT.sum()

In [ ]:
mg_loc_p_census.query("MZONA.isna()").POBTOT.sum()

In [ ]:
mg_ageb_census.MZONA.isna().sum()

In [ ]:
# For the actual zones
# taz_list = taz.index.drop(access_points)
taz_list = sorted(list(mtaz["ZONAEOD202"].unique()))
with PdfPages("../outputs/taz_validation/micro_to_zone_validation.pdf") as pdf:
    for z in taz_list:
        # if z != "02F":
        #    continue
        fig, axes = plt.subplots(1, 4, figsize=(40, 10), layout="constrained")

        mtaz_in_z = (
            mtaz.query("ZONAEOD202 == @z")
            .assign(ID_cat=lambda df: df.index)
            .sort_index()
        )
        color_dict = {
            zz: cc.cm.glasbey_hv(i) for i, zz in enumerate(mtaz_in_z["ID_cat"])
        }
        cmap = colors.ListedColormap(list(color_dict.values()))
        mtaz_in_z.plot(
            edgecolor="red",
            ax=axes[0],
            column="ID_cat",
            legend=True,
            cmap=cmap,
            categorical=True,
        )
        if z in taz.index:
            taz.loc[z:z].plot(ax=axes[0], facecolor="none", lw=3)
            taz.plot(ax=axes[3], facecolor="none", lw=1)
            taz.loc[z:z].plot(ax=axes[3], facecolor="grey", edgecolor="k", lw=2)
            taz.loc[z:z].plot(ax=axes[1], facecolor="none", lw=3, alpha=0.5)
            taz.loc[z:z].plot(ax=axes[2], facecolor="none", lw=3, alpha=0.5)
            axes[0].set_title(z)
        else:
            axes[0].set_title(f"{z} SIN ZONA")

        for zz, row in mtaz_in_z.iterrows():
            agebs_in_zz = agebs_zones.query("MZONA == @zz").sort_values("MZONA")
            agebs_in_zz.plot(ax=axes[1], color=color_dict[zz], edgecolor="black")

            agebs_mg_in_zz = mg_ageb_census.query("MZONA == @zz").sort_values("MZONA")
            agebs_mg_in_zz.plot(
                ax=axes[2], color=color_dict[zz], edgecolor="black", lw=2
            )

            locs_mg_in_zz = mg_loc_census_rural.query("MZONA == @zz").sort_values(
                "MZONA"
            )
            locs_mg_in_zz.plot(
                ax=axes[2], color=color_dict[zz], edgecolor="black", hatch="//", lw=2
            )

            locs_p_mg_in_zz = mg_loc_p_census.query("MZONA == @zz").sort_values("MZONA")
            locs_p_mg_in_zz.plot(
                ax=axes[2],
                color=color_dict[zz],
                edgecolor="black",
                markersize=200,
                lw=2,
            )

        mtaz_in_z.plot(
            facecolor="none",
            edgecolor="red",
            ax=axes[1],
            lw=2,
        )
        mtaz_in_z.plot(
            facecolor="none",
            edgecolor="red",
            ax=axes[2],
            lw=2,
        )

        axes[1].set_title("Agebs reportadas en encuesta.")
        axes[2].set_title("Agebs y Loc del MG.")

        pdf.savefig()
        plt.close()

In [ ]:
mtaz[["ZONAEOD202", "POBTOT"]].assign(
    AGEBS=mg_ageb_census.groupby("MZONA").POBTOT.sum(),
    LOC_R=mg_loc_census_rural.groupby("MZONA").POBTOT.sum(),
    LOC_P=mg_loc_p_census.groupby("MZONA").POBTOT.sum(),
    MG_TOT=lambda df: df.AGEBS.fillna(0) + df.LOC_R.fillna(0) + df.LOC_P.fillna(0),
    # MG_TOT=lambda df: df.AGEBS.fillna(0) + df.LOC_R.fillna(0),
    DIFF=lambda df: df.POBTOT - df.MG_TOT,
).pipe(lambda df: df[df.DIFF != 0])


In [ ]:
mg_loc_p_census.MZONA.isna().sum()

In [ ]:
mg_ageb_census

In [ ]:
# AGEBS that can be replaced by their locality to avoid censoring.
assert mg_ageb_census.MZONA.isna().sum() == 0
mg_ageb_census.groupby(["MUN", "LOC"]).MZONA.nunique().pipe(lambda s: s[s == 1]).shape

# Should still verify POBTOT matches.

In [ ]:
mg_ageb_census.query("MZONA.isna()").POBTOT.sum()

In [ ]:
(
    mg_loc_census_rural.query("MZONA.isna()").POBTOT.sum()
    + mg_loc_p_census.query("MZONA.isna()").POBTOT.sum()
)